In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

from last weeks notebook(was having trouble excuting the linear regression model because of how many columns encoded dataset was to big so I did not end up using this)

In [8]:
'''housing_feature = pd.read_csv('CRMLS_Preprocessed_FeatureSpecific.csv')

categorical_cols = housing_feature.select_dtypes(include="object").columns

housing_encoded = pd.get_dummies(housing_feature,columns=categorical_cols,drop_first=True)

print(housing_encoded.shape)

X = housing_encoded.drop(columns=["ClosePrice"])
y = housing_encoded["ClosePrice"]

#time base train/test split
latest_month = housing_encoded["DataMonth"].max()

train = housing_encoded[housing_encoded["DataMonth"] < latest_month]
test  = housing_encoded[housing_encoded["DataMonth"] == latest_month]

X_train = train.drop(columns=["ClosePrice"])
y_train = train["ClosePrice"]

X_test = test.drop(columns=["ClosePrice"])
y_test = test["ClosePrice"]

# Remove DataMonth after splitting since it is no longer needed as a predictor
X_train = X_train.drop(columns=["DataMonth"])
X_test = X_test.drop(columns=["DataMonth"])

numeric_features = X_train.select_dtypes(include="number").columns

scaler = StandardScaler()

X_train[numeric_features] = scaler.fit_transform(X_train[numeric_features])
X_test[numeric_features] = scaler.transform(X_test[numeric_features])'''

'housing_feature = pd.read_csv(\'CRMLS_Preprocessed_FeatureSpecific.csv\')\n\ncategorical_cols = housing_feature.select_dtypes(include="object").columns\n\nhousing_encoded = pd.get_dummies(housing_feature,columns=categorical_cols,drop_first=True)\n\nprint(housing_encoded.shape)\n\nX = housing_encoded.drop(columns=["ClosePrice"])\ny = housing_encoded["ClosePrice"]\n\n#time base train/test split\nlatest_month = housing_encoded["DataMonth"].max()\n\ntrain = housing_encoded[housing_encoded["DataMonth"] < latest_month]\ntest  = housing_encoded[housing_encoded["DataMonth"] == latest_month]\n\nX_train = train.drop(columns=["ClosePrice"])\ny_train = train["ClosePrice"]\n\nX_test = test.drop(columns=["ClosePrice"])\ny_test = test["ClosePrice"]\n\n# Remove DataMonth after splitting since it is no longer needed as a predictor\nX_train = X_train.drop(columns=["DataMonth"])\nX_test = X_test.drop(columns=["DataMonth"])\n\nnumeric_features = X_train.select_dtypes(include="number").columns\n\nscaler =

In [9]:
housing_feature = pd.read_csv( "CRMLS_Preprocessed_FeatureSpecific.csv")

print("Original shape:", housing_feature.shape)


#removing categorical columns that have too mnay unique values
categorical_cols = housing_feature.select_dtypes(include=["object", "bool"]).columns.tolist()

# needed for train test split
if "DataMonth" in categorical_cols:
    categorical_cols.remove("DataMonth")

high_cardinality_cols = [col for col in categorical_cols if housing_feature[col].nunique(dropna=True) > 100]

print("\nHigh-cardinality columns being dropped:")
print(high_cardinality_cols)

housing_model = housing_feature.drop(columns=high_cardinality_cols,errors="ignore")

#converting remaining categorical vars into dummy variables
categorical_cols = housing_model.select_dtypes(include=["object", "bool"]).columns.tolist()

if "DataMonth" in categorical_cols:
    categorical_cols.remove("DataMonth")

housing_encoded = pd.get_dummies(housing_model,columns=categorical_cols,drop_first=True,dtype="int8")

print("\nEncoded shape:", housing_encoded.shape)


# train test split part
latest_month = housing_encoded["DataMonth"].max()

train = housing_encoded[housing_encoded["DataMonth"] < latest_month].copy()

test = housing_encoded[housing_encoded["DataMonth"] == latest_month].copy()

print("\nLatest test month:", latest_month)
print("Training pool rows:", len(train))
print("Test rows:", len(test))

#only using a sample of the training data to make the basline fast
#while also keeping the full test month
sample_size = min(30000, len(train))

train_sample = train.sample(n=sample_size,random_state=42)

X_train = train_sample.drop(columns=["ClosePrice", "DataMonth"],errors="ignore")

y_train = train_sample["ClosePrice"]

X_test = test.drop(columns=["ClosePrice", "DataMonth"],errors="ignore")

y_test = test["ClosePrice"]

print("\nSampled training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Number of model features:", X_train.shape[1])

#hanfle any reamming inf or missing numeric values if any
numeric_features = X_train.select_dtypes(include=np.number).columns

X_train[numeric_features] = X_train[numeric_features].replace([np.inf, -np.inf],np.nan)

X_test[numeric_features] = X_test[numeric_features].replace([np.inf, -np.inf],np.nan)

for col in numeric_features:
  median_value = X_train[col].median()
  X_train[col] = X_train[col].fillna(median_value)
  X_test[col] = X_test[col].fillna(median_value)

#standar scaler

scaler = StandardScaler()

X_train[numeric_features] = scaler.fit_transform(X_train[numeric_features])
X_test[numeric_features] = scaler.transform(X_test[numeric_features])


Original shape: (141892, 54)

High-cardinality columns being dropped:
['Flooring', 'CloseDate', 'MLSAreaMajor', 'ElementarySchool', 'BuilderName', 'SubdivisionName', 'City', 'ContractStatusChangeDate', 'ListingContractDate', 'MiddleOrJuniorSchool', 'HighSchool', 'LotSizeDimensions', 'HighSchoolDistrict', 'PostalCode']

Encoded shape: (141892, 280)

Latest test month: 202605
Training pool rows: 129875
Test rows: 12017

Sampled training rows: 30000
Testing rows: 12017
Number of model features: 278


In [10]:
#creating and training model
model = LinearRegression()
model.fit(X_train,y_train)

#model prediction
y_pred = model.predict(X_test)
r2 = r2_score(y_test,y_pred)

print(f"Linear Regression R2: {r2:}")

rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"RMSE: {rmse}\n")

#features using:
print(X_train.shape)

#training rows used
print(f"Training rows used:: {len(X_train)}")

#testing rows used
print(f"\nTesting rows used:: {len(X_test)}")


Linear Regression R2: 0.04828527513685266
RMSE: 1637438.169688876

(30000, 278)
Training rows used:: 30000

Testing rows used:: 12017


Week 5

In [11]:
tree = DecisionTreeRegressor(max_depth=10,andom_state=2026)

tree.fit(X_train, y_train)

tree_pred = tree.predict(X_test)

tree_r2 = r2_score(y_test, tree_pred)
tree_rmse = np.sqrt(mean_squared_error(y_test, tree_pred))

print("Decision Tree R²:", tree_r2)
print("Decision Tree RMSE:", tree_rmse)

Decision Tree R²: -17.157750417103315
Decision Tree RMSE: 7152247.533549739


In [12]:
forest = RandomForestRegressor(n_estimators=100,random_state=2026,n_jobs=-1)

forest.fit(X_train, y_train)

forest_pred = forest.predict(X_test)

forest_r2 = r2_score(y_test, forest_pred)
forest_rmse = np.sqrt(mean_squared_error(y_test, forest_pred))

print("Random Forest R²:", forest_r2)
print("Random Forest RMSE:", forest_rmse)

Random Forest R²: -2.594075889497205
Random Forest RMSE: 3182036.3537016395


Linear Regression achieved the highest test R2 of 0.048.

Both the Decision Tree and Random Forest produced negative R2 values they performed worse than simply predicting the mean sale price.

Needs to do more tuning and feature engineering

In [13]:
results = pd.DataFrame({
    "Model":[
        "Linear Regression",
        "Decision Tree",
        "Random Forest"],
    "R2":[
        r2,
        tree_r2,
        forest_r2],
    "RMSE":[
        rmse,
        tree_rmse,
        forest_rmse]})

print(results)

               Model         R2          RMSE
0  Linear Regression   0.048285  1.637438e+06
1      Decision Tree -17.157750  7.152248e+06
2      Random Forest  -2.594076  3.182036e+06
